# Launching multiple agenst in parallel

import libraries

In [1]:
import json
import re
from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass, field
import anthropic

import os
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY")
client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY) 

In [4]:
# function to parse JSON from a string, handling code blocks and extracting the JSON content
def _parse_json(text: str) -> dict:
    text = text.strip()
    m = re.search(r'```(?:json)?\s*\n?([\s\S]+?)\n?\s*```', text)
    if m:
        text = m.group(1).strip()
        start, end = text.find('{'), text.rfind('}')+1
    if start != -1 and end > start:
        text = text[start:end]
    return json.loads(text)

Define the handoff packets

In [5]:
@dataclass
class HandoffPacket:
    context: str
    task: str

def build_financial_handoff(company: str, fiscal_year: int) -> HandoffPacket:
    context = f"Due diligence for {company}, fiscal year {fiscal_year}."
    task = (
        f"Summarize the financial profile of {company} for FY {Fiscal_year} based on "
        "publicly available information. Return a JSON object with these keys: "
        "revenue_estimate (string or null), growth_signal ('positive'/'neutral'/'negative'), "
        "and risk_signal (list of strings), data_gaps (list of strings), sources (list of strings). "
        "Return only the JSON object."
    )
    return HandoffPacket(context=context, task=task)

def build_legal_handoff(company: str) -> HandoffPacket:
    context = f"Legal Compliance for {company}." 
    task = (
        f"Access the regulatory and legal standing of {company} based on publicly available "
        "information. Return a JSON object with these keys: "
        "compliance_signal ('clean'/'caution'/'unknown'), nores (list of strings), "
        "data_gaps (list of strings), sources (list of strings). "
        "Do not invent specific case numbers. Return only the JSON object."
    )
    return HandoffPacket(context=context, task=task)

def build_news_handoff(company: str, days: int = 90) -> HandoffPacket:
    context = f"News sentiment for {company}, last {days}." 
    task = (
        f"Summarize public news sentiment for company {company} over the past {days} days. "
        "Return only the JSON object with these keys: "
        "sentiment ('positive'/'neutral'/'negative'), highlights (list of up to 3 strings), "
        "data_gaps (list of strings), sources (list of strings). "
        "Return only the JSON object."
    )
    return HandoffPacket(context=context, task=task)
    
def packet_to_prompt(packet: HandoffPacket) -> str:
    return f"Context: {packet.context}\nTask: {packet.task}"



In [ ]:
# Define the specialists
SPECIALIST_SYSTEMS = {
    "financial": "You are a financial due diligence specialist. Return structured JSON only.",
    "legal": "You are a legal compliance specialist. Return structured JSON only.",
    "news": "You are a news sentiment analyst. Return structured JSON only."
}

model =  "claude-sonnet-5-5"
def run_specialist(system: str, prompt:str) -> dict:
    response = client.completions.create(
        model=model,
        system=system,
        max_tokens=1024,
        messages=[{"role":"user","content":prompt}]
    )
    for block in response.content:
        if block.type == "text":
            return _parse_json(block.text)
    return {"error":"no_text_block", "data_gaps":["specialist returned no content"]}